## 0. Descriptive Analytics

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
RAW_DATA_PATH = Path('../data/raw/')
revenue = pd.read_csv(RAW_DATA_PATH / 'company_quarterly_revenue_raw.csv')
orders = pd.read_csv(RAW_DATA_PATH / 'orders-1.csv')
shipments = pd.read_csv(RAW_DATA_PATH / 'shipments.csv')

<b> Revenue </b>

In [ ]:
# shape
display(revenue.shape)

# first few rows
display(revenue.head())

The revenue data set is not tidy because it violates the rule that each variable must be represented by a single column.
In this case, the quarterly values are represented as column headers when instead, we should have a quarters column with those headers as values.

In [ ]:
sns.barplot(data=revenue, x='company', y='2023-Q1', errorbar=None, estimator='sum')
plt.title('Revenue by Company - Quarter: 2023-Q1')
plt.ylabel('Revenue')
plt.show()

<b> Orders </b>

In [ ]:
# shape
display(orders.shape)

# first few rows
display(orders.head())

The orders data set is tidy because it adheres to the three rules of tidy data. Each variable has it's own column, each observation has it's own row and each value has it's own cell.

In [ ]:
sns.barplot(data=orders, x='company', y='amount', errorbar=None, estimator='sum')
plt.title('Total Order Amount by Company')
plt.show()

<b> Shipments </b>

In [ ]:
# shape
display(shipments.shape)

# first few rows
display(shipments.head())

The shipments data set is tidy because it adheres to the three rules of tidy data. Each variable has it's own column, each observation has it's own row and each value has it's own cell.

In [ ]:
sns.countplot(data=shipments, y= 'order_id')
plt.title('Number of shipments by order_id')
plt.show()

## 1. Revenue Table

In [ ]:
revenue

The revenue data set is not tidy because it violates the rule that each variable must be represented by a single column.
In this case, the quarterly values are represented as column headers when instead, we should have a quarters column with those headers as values.

In [ ]:
revenue_tidy = revenue.melt(id_vars='company', var_name='quarter', value_name='revenue')
revenue_tidy.head()

## 2. Duplicate Keys

In [ ]:
any(orders['order_id'].duplicated())

In [ ]:
any(shipments['order_id'].duplicated())

order_id is unique in orders but not shipments. This tells me that the join relationship is one-to-many.

## 3. Join orders and shipments correctly

In [ ]:
orders_and_shipments = orders.merge(right=shipments, how='inner', on='order_id', validate='1:m')
orders_and_shipments.head()

In [ ]:
# mean amount per customer - orders_and_shipments table

orders_and_shipments.groupby('customer_id')['amount'].mean()

In [ ]:
# mean amount per customer - orders table

orders.groupby('customer_id')['amount'].mean()

The mean amounts per customers differ. 
This is because in the joined table, amounts are getting duplicated whereas with the orders table, 
there is no duplication. 
The joined result is distorting the mean by weighting orders with more shipments more heavily. This yields an incorrect result so the values
from the orders table should be used as each amount is represented once.

## 4. Produce one analysis-ready table

In [ ]:
display(any(orders_and_shipments.duplicated(subset=['company', 'quarter'])))
orders_and_shipments.head()

In [ ]:
display(any(revenue_tidy.duplicated(subset=['company', 'quarter'])))
revenue_tidy.head()

In [ ]:
results = orders_and_shipments.merge(revenue_tidy, on=['company', 'quarter'], how='inner', validate='m:1').reset_index(drop=True)
results.head() # each row represents a shipment enriched with revenue and order info

In [ ]:
results.shape[0]

In [ ]:
assert results.shape[0] == 35, 'Expected row count of 35 not produced.'

## 5. Visualization

In [ ]:
# one row per company-quarter for visualization
deduplicated_results = results.drop_duplicates(subset=['company', 'quarter'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# left chart: task 0 char
sns.barplot(data=revenue, x='company', y='2023-Q1', errorbar=None, estimator='sum', order=sorted(revenue['company'].unique()), ax=axes[0])
axes[0].set_title('Revenue by Company - 2023-Q1')
axes[0].set_ylabel('Revenue')

# right chart: task 5 chart
sns.barplot(data=deduplicated_results, x='company', y='revenue', hue='quarter', errorbar=None, order=sorted(deduplicated_results['company'].unique()), ax=axes[1])
axes[1].set_title('Revenue by Company and Quarter')
axes[1].set_ylabel('Revenue')

axes[1].legend(
    title='quarter',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

Tidying the revenue data made it easier to compare revenue across multiple companies using the quarter variable as a hue.
Previously, each quarter was stored in a separate column, requiring hardcoding of the quarers to be plotted.
The joined table also allows us to idenfity which compnay-quarter comboes had matching order and shipment records.